# Video exposure

Fig. 2(a). Match platform-labeled AIGC videos to unlabeled videos, then compare 31-day exposure and days to 90% of that exposure.

### 1. The two arms

Focal is the treated arm. Comparison is the other arm. Nothing is paired yet.

In [1]:
from pathlib import Path
import sys

here = Path.cwd().resolve()
for candidate in [here, *here.parents]:
    if (candidate / "run_pipeline.py").exists():
        sys.path.insert(0, str(candidate))
        root = candidate
        break
else:
    raise FileNotFoundError("Open this notebook from the repository.")

from core.pipeline import ensure_synthetic, load_config

cfg = load_config(root)
spec = cfg["tasks"]["video"]
pool = ensure_synthetic(root, cfg, rebuild=False)[{"creator": "creators", "consumer": "consumers", "video": "videos", "algorithm": "algorithm"}["video"]]
focal = pool[pool["treated"] == 1].reset_index(drop=True)
comparison = pool[pool["treated"] == 0].reset_index(drop=True)
import pandas as pd
pd.DataFrame([{
    "focal_label": spec["focal"],
    "comparison_label": spec["comparison"],
    "n_focal": len(focal),
    "n_comparison": len(comparison),
    "exact": ", ".join(spec["exact"]),
    "caliper": ", ".join(f"{name} <= {limit}" for name, limit in (spec.get("caliper") or {}).items()) or "none",
}])


,focal_label,comparison_label,n_focal,n_comparison,exact,caliper
0,platform-labeled AIGC,unlabeled content,1200,1200,"category, city, upload_week",none


### 2. Exact cells

Units are blocked on the categorical covariates first. A focal unit whose cell has no comparison unit cannot be matched, before any distance is computed.

In [2]:
from core.matching import exact_cell_counts

cells = exact_cell_counts(focal, comparison, spec["exact"])
cells.sort_values(["n_comparison", "n_focal"], ascending=[True, False])


,category,city,upload_week,n_focal,n_comparison
3,food,east,week_1,28,15
4,food,north,week_2,23,15
23,leisure,south,week_1,16,16
14,leisure,east,week_4,30,17
21,leisure,south,week_3,21,17
36,travel,east,week_2,32,19
38,travel,east,week_4,23,19
44,travel,south,week_2,26,20
8,food,south,week_2,24,20
20,leisure,south,week_2,23,20


### 3. Scale of the continuous covariates

Nearest neighbor uses a pooled z-score. The mean and standard deviation are taken over both arms together.

In [3]:
from core.matching import covariate_scale

covariate_scale(focal, comparison, spec["numeric"])


,covariate,pooled_mean,pooled_sd
0,uploads_14d,3.501946,1.258782
1,uploads_60d,8.696302,3.092976
2,plays_14d,98.634806,41.187213
3,plays_60d,296.874580,126.251764
4,watch_time_14d,179.423003,74.495756
5,watch_time_60d,538.329929,230.029864
6,followers,865.239557,644.640176


### 4. Nearest neighbor inside the cell

Each focal unit takes the comparison unit in the same cell with the smallest Euclidean distance. A caliper, when there is one, sets the distance to infinity and that neighbor is dropped. The ledger counts, per cell, how many focal units were kept and how many were dropped.

In [4]:
from core.matching import match_process

pairs, ledger = match_process(
    focal,
    comparison,
    spec["exact"],
    spec["numeric"],
    spec["outcomes"],
    spec.get("caliper") or {},
)
ledger.sort_values(["n_dropped", "n_focal"], ascending=[False, False])


,category,city,upload_week,n_focal,n_comparison,n_matched,n_dropped,median_distance
16,leisure,north,week_2,38,24,38,0,1.791545
6,food,north,week_4,34,29,34,0,1.737506
30,beauty,north,week_4,34,26,34,0,1.671215
34,beauty,south,week_4,33,28,33,0,1.618918
36,travel,east,week_2,32,19,32,0,1.920225
5,food,north,week_3,31,23,31,0,1.619469
11,food,south,week_1,30,29,30,0,1.598780
14,leisure,east,week_4,30,17,30,0,1.903002
45,travel,south,week_3,30,31,30,0,1.673488
31,beauty,north,week_1,29,26,29,0,1.557935


### 5. What was dropped, and how often a comparison unit is reused

Replacement means one comparison unit may be the neighbor for more than one focal unit.

In [5]:
import pandas as pd

reuse = pairs["comparison_row"].value_counts()
pd.DataFrame([{
    "focal_units": len(focal),
    "comparison_units": len(comparison),
    "exact_cells": len(ledger),
    "dropped_no_comparison_in_cell": int(ledger.loc[ledger["n_comparison"] == 0, "n_focal"].sum()),
    "dropped_by_caliper_or_empty_neighborhood": int(ledger.loc[ledger["n_comparison"] > 0, "n_dropped"].sum()),
    "pairs": len(pairs),
    "comparison_units_used": int(reuse.shape[0]),
    "max_reuse": int(reuse.max()) if len(reuse) else 0,
}])


,focal_units,comparison_units,exact_cells,dropped_no_comparison_in_cell,dropped_by_caliper_or_empty_neighborhood,pairs,comparison_units_used,max_reuse
0,1200,1200,48,0,0,1200,669,10


### 6. Eight pairs

Exact covariates are copied onto both sides. They agree by construction. `distance` is the z-score Euclidean distance of the chosen neighbor.

In [6]:
show = ["distance"]
for column in spec["exact"]:
    show += [f"{column}_focal", f"{column}_comparison"]
for column in spec.get("caliper") or {}:
    gap = f"{column}_gap"
    pairs[gap] = (pairs[f"{column}_focal"] - pairs[f"{column}_comparison"]).abs()
    show.append(gap)
pairs[show].head(8)


,distance,category_focal,category_comparison,city_focal,city_comparison,upload_week_focal,upload_week_comparison
0,1.661181,food,food,east,east,week_2,week_2
1,2.792371,food,food,east,east,week_2,week_2
2,1.637270,food,food,east,east,week_2,week_2
3,2.154506,food,food,east,east,week_2,week_2
4,1.672478,food,food,east,east,week_2,week_2
5,2.862768,food,food,east,east,week_2,week_2
6,3.198986,food,food,east,east,week_2,week_2
7,1.980504,food,food,east,east,week_2,week_2


### 7. Balance before and after

Standardized mean difference of each covariate, on the raw arms and on the matched pairs.

In [7]:
from core.balance import balance_before_after

balance = balance_before_after(focal, comparison, pairs, list(spec["exact"]) + list(spec["numeric"]))
balance


,covariate,mean_focal_before,mean_comparison_before,standardized_difference_before,mean_focal_after,mean_comparison_after,standardized_difference_after
0,followers,851.922432,878.556682,-0.041308,851.922432,784.948733,0.119308
1,plays_14d,97.866012,99.403599,-0.037323,97.866012,94.538442,0.091162
2,plays_60d,301.849300,291.899861,0.078835,301.849300,284.760149,0.148238
3,uploads_14d,3.519747,3.484146,0.028273,3.519747,3.400295,0.103103
4,uploads_60d,8.606482,8.786122,-0.058080,8.606482,8.524022,0.029305
5,watch_time_14d,177.594524,181.251482,-0.049084,177.594524,171.983415,0.084514
6,watch_time_60d,535.929848,540.730011,-0.020860,535.929848,520.246308,0.073464


### 8. The contrast

Mean of focal minus comparison. The standard error is Abadie-Imbens, with up to five within-arm neighbors. The Hodges-Lehmann median and the Wilcoxon p-value are the rank summary beside that mean.

In [8]:
from core.imbens import contrast_table

effects = contrast_table(
    focal,
    comparison,
    pairs,
    spec["outcomes"],
    spec["exact"],
    spec["numeric"],
    n_neighbors=int(cfg.get("imbens_neighbors", 5)),
)
out = root / "data" / "results"
out.mkdir(parents=True, exist_ok=True)
pairs.to_csv(out / "video_pairs.csv", index=False)
effects.to_csv(out / "video_effects.csv", index=False)
balance.to_csv(out / "video_balance.csv", index=False)
effects


,outcome,n_pairs,hl_focal_minus_comparison,mean_difference,wilcoxon_statistic,wilcoxon_p,variance,standard_error,z_statistic,p_value,ci95_low,ci95_high
0,exposure_31d,1200,-179.799979,-179.747131,0.0,8.134563e-198,4.804842,2.191995,-82.001618,0.0,-184.043441,-175.450821
1,days_to_90,1200,-4.893320,-4.903253,370.0,2.051768e-197,0.005972,0.077279,-63.448580,0.0,-5.054721,-4.751786
